## 1. 岛屿数量

- **难度**：中等  
- **标签**：图论、广度优先搜索（BFS）、网格

**题目**：给你一个由 '1'（陆地）和 '0'（水）组成的二维网格，请你计算网格中岛屿的数量。岛屿总是被水包围，并且每座岛屿只能由水平方向和/或竖直方向上相邻的陆地连接形成。此外，你可以假设该网格的四条边均被水包围。

**示例**：
```
输入：grid = [
  ["1","1","1","1","0"],
  ["1","1","0","1","0"],
  ["1","1","0","0","0"],
  ["0","0","0","0","0"]
]
输出：1

输入：grid = [
  ["1","1","0","0","0"],
  ["1","1","0","0","0"],
  ["0","0","1","0","0"],
  ["0","0","0","1","1"]
]
输出：3
```

**思路**：把网格看成图，每个陆地格是节点，上下左右相邻格是边。扫描整个网格，每遇到一个 '1' 就是发现了一座新岛屿（计数 +1），然后以它为起点 BFS，把整座岛的陆地全部就地改成 '0'（"淹掉"），保证同一座岛不会被重复计数。时间 O(m×n)（每格最多访问常数次），空间 O(min(m, n))（队列最大长度，网格本身被就地修改无需 visited）。

**亮点**：「发现即淹没」——用就地标记（grid 改 '0'）代替 visited 数组，空间省到极致；BFS 采用「入队时立即标记」而非出队时，避免同一节点重复入队导致超时。


In [ ]:
from collections import deque
from typing import List

class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
        if not grid:
            return 0

        rows = len(grid)
        cols = len(grid[0])
        islands = 0

        # 上、下、左、右
        directions = [
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1)
        ]

        # BFS：把一整座岛屿访问完
        def bfs(start_r, start_c):
            queue = deque([(start_r, start_c)])

            # 标记为已经访问
            grid[start_r][start_c] = '0'

            while queue:
                r, c = queue.popleft()

                # 检查上下左右
                for dr, dc in directions:
                    new_r = r + dr
                    new_c = c + dc

                    # 如果没有越界，并且是陆地
                    if (
                        0 <= new_r < rows
                        and 0 <= new_c < cols
                        and grid[new_r][new_c] == '1'
                    ):
                        # 标记为已访问
                        grid[new_r][new_c] = '0'

                        # 加入队列，继续搜索
                        queue.append((new_r, new_c))

        # 遍历整个网格
        for r in range(rows):
            for c in range(cols):

                # 发现新的岛屿
                if grid[r][c] == '1':
                    islands += 1
                    bfs(r, c)

        return islands


# 测试
sol = Solution()
grid1 = [row[:] for row in [
    ["1","1","1","1","0"],
    ["1","1","0","1","0"],
    ["1","1","0","0","0"],
    ["0","0","0","0","0"]
]]
print(sol.numIslands(grid1))  # 1

grid2 = [row[:] for row in [
    ["1","1","0","0","0"],
    ["1","1","0","0","0"],
    ["0","0","1","0","0"],
    ["0","0","0","1","1"]
]]
print(sol.numIslands(grid2))  # 3


## 2. 课程表

- **难度**：中等  
- **标签**：图论、拓扑排序、广度优先搜索（BFS）

**题目**：你这个学期必须选修 numCourses 门课程，记为 0 到 numCourses - 1。在选修某些课程之前需要一些先修课程，先修课程按数组 prerequisites 给出，其中 prerequisites[i] = [ai, bi] 表示如果要学习课程 ai 则必须先学习课程 bi。请你判断是否可能完成所有课程的学习？如果可以返回 true，否则返回 false。

**示例**：
```
输入：numCourses = 2, prerequisites = [[1, 0]]
输出：true
解释：共有 2 门课程。学习课程 1 之前，你需要完成课程 0。这是可能的。

输入：numCourses = 2, prerequisites = [[1, 0], [0, 1]]
输出：false
解释：共 2 门课程。学习课程 1 之前需先完成 0，学习 0 之前需先完成 1，互相依赖成环，不可能完成。
```

**思路**：把依赖建成有向图 b → a（先学 b 才能学 a），同时统计每门课的入度（先修课数量）。Kahn 拓扑排序（BFS 实现）：入度为 0 的课程没有未满足的先修要求，直接入队"学习"；每学完一门，它指向的后继课程入度减 1，入度归零即"解锁"入队。最终若 learned == numCourses 说明所有课程都能修完（无环），否则图中存在循环依赖。时间 O(V + E)，空间 O(V + E)。

**亮点**：拓扑排序把"能否完成"转化为"能否遍历完"——环上课程的入度永远无法归零，learned 计数自然缺失，无需显式找环；"入度归零即解锁"像流水线放行，是 DAG 调度类问题（编译依赖、任务编排）的通用范式。


In [ ]:
from collections import deque
from typing import List

class Solution:
    def canFinish(self, numCourses: int, prerequisites: List[List[int]]) -> bool:
        graph = [[] for _ in range(numCourses)]
        indegree = [0] * numCourses

        # b -> a：学 a 之前必须先学 b
        for a, b in prerequisites:
            graph[b].append(a)
            indegree[a] += 1

        # 没有先修课的课程可以直接学习
        queue = deque(i for i in range(numCourses) if indegree[i] == 0)

        learned = 0

        while queue:
            course = queue.popleft()
            learned += 1

            # 学完 course，依赖它的课程少一个先修要求
            for next_course in graph[course]:
                indegree[next_course] -= 1

                if indegree[next_course] == 0:
                    queue.append(next_course)

        return learned == numCourses


# 测试
sol = Solution()
print(sol.canFinish(2, [[1, 0]]))                          # True
print(sol.canFinish(2, [[1, 0], [0, 1]]))                  # False（成环）
print(sol.canFinish(6, [[1, 0], [2, 0], [3, 1], [4, 1], [5, 3]]))  # True（树形依赖）
